<a href="https://colab.research.google.com/github/reinaldoyungh/Python/blob/main/IA_LISTA_PERGUNTAS_E_RESPOSTAS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Manipulando dados com PANDAS

In [1]:
import pandas as pd

In [2]:
df = pd.read_csv("/content/meu_csv.csv")

In [3]:
df

,nome,media
0,Ana,7.5
1,Carlos,8.2
2,Beatriz,9.1
3,Diogo,6.8
4,Eduarda,5.9
5,Fernanda,8.5
6,Gustavo,7.3
7,Helena,9.5
8,Igor,6.1
9,Juliana,8.9


Desafio:

1. Criar um arquivo .txt a partir de uma lista de perguntas vindas de uma lista em Python [ ].

2. Ler essas perguntas desse arquivo .txt.

3. Obter respostas de um LLM para cada uma.

4. Salvar os resultados em um novo arquivo .csv

5. Ler o arquivo .csv usando Pandas

In [18]:
import google.generativeai as genai
from google.colab import userdata

# 1. Configuração da API (Recuperando a chave do gerenciador de segredos do Colab)
genai.configure(api_key=userdata.get('GEMINI_API_KEY'))
model = genai.GenerativeModel('gemini-2.5-flash')

In [1]:
!pip install -q groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.7/141.7 kB 4.8 MB/s eta 0:00:00


In [2]:
import os
from google.colab import userdata

os.environ["GROQ_API_KEY"] = userdata.get('GROQ_API_KEY')

In [3]:
from groq import Groq
client = Groq()

In [12]:
lista_de_perguntas = [
    "Qual o alimento mais consumido pelo brasileiro?",
    "Qual o prato preferido pelo Brasileiro?"]

with open('lista-de-perguntas.txt', 'w',encoding="utf-8") as arquivo:
  for perguntas in lista_de_perguntas:
    arquivo.write(perguntas + '\n')

In [5]:
#def chama_IA(perguntas):
#
#    for retorno in perguntas:
#        respostas = model.generate_content(
#            contents = perguntas
#        )
#
#    return respostas.text

In [9]:
def chama_IA(perguntas):
    all_responses = []
    for pergunta_individual in perguntas:
        response_stream = client.chat.completions.create(
          model="openai/gpt-oss-20b",
          messages=[
            {
              "role": "user",
              "content": pergunta_individual
            }
          ],
          temperature=1,
          max_completion_tokens=1024,
          top_p=1,
          reasoning_effort="medium",
          stream=True,
          stop=None
        )

        collected_text = ""
        for chunk in response_stream:
            if chunk.choices and chunk.choices[0].delta.content:
                collected_text += chunk.choices[0].delta.content
        all_responses.append(collected_text)
    return all_responses

In [15]:
nova_lista_leitura = []
respostas = []
with open('lista-de-perguntas.txt', 'r',encoding="utf-8") as arquivo:
  for linha in arquivo:
    nova_lista_leitura.append(linha.strip())
    respostas = chama_IA(nova_lista_leitura)
    #print(nova_lista_leitura)
    #print(respostas)

    #Com as listas nova_lista_leitura e respostas, gere um novo arquivo chamado lista-de-perguntas-e-respostas.txt
    with open('lista-de-perguntas-e-respostas.txt', 'w',encoding="utf-8") as arquivo:
      for pergunta, resposta in zip(nova_lista_leitura, respostas):
        arquivo.write(f"Pergunta: {pergunta}\nResposta: {resposta}\n\n")


